<div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);padding:32px 28px;border-radius:12px;margin-bottom:8px">
<h1 style="color:#e9c46a;margin:0 0 6px">📊 FIFA World Cup 2026</h1>
<h2 style="color:#a8dadc;font-weight:400;margin:0 0 14px">Notebook 1 of 4 — Exploratory Data Analysis</h2>
<p style="color:#ccc;line-height:1.6;margin:0 0 16px">
Deep dive into 150+ years of World Cup history. We explore scoring trends,
match outcomes, team dominance, and tactical patterns — before touching any model.
</p>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#2a9d8f">📊 EDA</span>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#457b9d;margin-left:6px">🗃️ 3 Datasets</span>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#6a4c93;margin-left:6px">📈 9 Visualizations</span>
</div>

---
### 📋 This notebook covers
| Section | Description |
|---------|-------------|
| 1. Setup | Imports, palette, display config |
| 2. Load Data | WorldCupMatches, WorldCups, results.csv |
| 3. EDA | Goals over time, outcomes, top teams, efficiency, knockout vs group |

> **Next:** `02_feature_engineering.ipynb` — build ELO ratings, rolling form, head-to-head features


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"]  = 13
plt.rcParams["axes.labelsize"]  = 11

PALETTE = ["#2a9d8f","#e9c46a","#e76f51","#264653","#a8dadc",
           "#457b9d","#fca311","#6a4c93","#f4a261","#1d3557"]
print("✅ Imports ready")

## 2. Load Data
Three source files — all placed in the `data/` folder.

In [ ]:
# ── WorldCupMatches: every WC match 1930–2022 ────────────────────────────────
wc_df = pd.read_csv("data/WorldCupMatches.csv")
wc_df = wc_df.dropna(subset=["Year"]).reset_index(drop=True)
wc_df["Year"] = wc_df["Year"].astype(int)
wc_df.columns = [c.lower().replace(" ","_") for c in wc_df.columns]
wc_df.drop(columns=["win_conditions","attendance","referee",
                     "assistant_1","assistant_2","roundid","matchid"],
           inplace=True, errors="ignore")
print(f"WorldCupMatches : {wc_df.shape[0]:,} rows × {wc_df.shape[1]} cols")

# ── WorldCups: tournament-level summaries ─────────────────────────────────────
cups_df = pd.read_csv("data/WorldCups.csv")
print(f"WorldCups       : {cups_df.shape[0]} tournaments")

# ── results.csv: all international matches since 1872 ────────────────────────
intl = pd.read_csv("data/results.csv")
intl["date"] = pd.to_datetime(intl["date"])
intl = intl.sort_values("date").reset_index(drop=True)
historical = intl[intl["home_score"].notna()].copy()
print(f"results.csv     : {len(intl):,} rows  ({historical['date'].min().year}–{historical['date'].max().year})")

In [ ]:
# Quick shape overview
print("=== Dataset shapes ===")
for name, df in [("wc_df", wc_df), ("cups_df", cups_df), ("intl", intl)]:
    print(f"  {name:12s}: {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols")
wc_df.head(3)

## 3. Exploratory Data Analysis

### 3.1 Derived Columns for EDA

In [ ]:
wc_df["result"]            = np.where(wc_df["home_team_goals"] > wc_df["away_team_goals"], "Home Win",
                             np.where(wc_df["home_team_goals"] < wc_df["away_team_goals"], "Away Win", "Draw"))
wc_df["total_goals"]       = wc_df["home_team_goals"] + wc_df["away_team_goals"]
wc_df["goal_difference"]   = wc_df["home_team_goals"] - wc_df["away_team_goals"]
wc_df["home_clean_sheet"]  = (wc_df["away_team_goals"] == 0).astype(int)
wc_df["away_clean_sheet"]  = (wc_df["home_team_goals"] == 0).astype(int)
wc_df["is_knockout"]       = (~wc_df["stage"].str.startswith("Group", na=False) &
                               ~wc_df["stage"].str.contains("Preliminary|First round", case=False, na=False))
wc_df["decade"]            = (wc_df["year"] // 10) * 10
wc_df["margin"]            = wc_df["goal_difference"].abs()
try:
    wc_df["half_time_goal_diff"] = wc_df["half-time_home_goals"] - wc_df["half-time_away_goals"]
except: pass

print(f"Derived columns added. Final shape: {wc_df.shape}")
wc_df[["home_team_name","away_team_name","home_team_goals",
       "away_team_goals","result","total_goals","is_knockout"]].head(5)

### 3.2 Dataset Overview & Missing Values

In [ ]:
print("Shape:", wc_df.shape)
print("\nMissing values (non-zero only):")
mv = wc_df.isnull().sum()
print(mv[mv > 0].to_string())
print("\nNumeric summary:")
wc_df[["home_team_goals","away_team_goals","total_goals"]].describe().round(2)

### 3.3 Goals Over Time

In [ ]:
goals_yr = wc_df.groupby("year").agg(
    total_goals=("total_goals","sum"),
    avg_goals=("total_goals","mean"),
    n_matches=("total_goals","count")
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(goals_yr["year"], goals_yr["total_goals"], marker="o", color=PALETTE[0], linewidth=2)
axes[0].fill_between(goals_yr["year"], goals_yr["total_goals"], alpha=0.15, color=PALETTE[0])
axes[0].set_title("Total goals scored per World Cup")
axes[0].set_ylabel("Total goals"); axes[0].set_xlabel("Year")

axes[1].plot(goals_yr["year"], goals_yr["avg_goals"], marker="s", color=PALETTE[2], linewidth=2)
axes[1].fill_between(goals_yr["year"], goals_yr["avg_goals"], alpha=0.15, color=PALETTE[2])
axes[1].axhline(goals_yr["avg_goals"].mean(), color="gray", linestyle="--", alpha=0.6,
                label=f'All-time avg = {goals_yr["avg_goals"].mean():.2f}')
axes[1].set_title("Average goals per match over time")
axes[1].set_ylabel("Avg goals / match"); axes[1].set_xlabel("Year"); axes[1].legend()

plt.suptitle("WC Goal Trends (1930–2022)", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()
goals_yr.tail(6)

### 3.4 Match Outcomes & Goal Distribution

In [ ]:
home_w = int((wc_df["home_team_goals"] > wc_df["away_team_goals"]).sum())
away_w = int((wc_df["home_team_goals"] < wc_df["away_team_goals"]).sum())
draws  = int((wc_df["home_team_goals"] == wc_df["away_team_goals"]).sum())
n = len(wc_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bars = axes[0].bar(["Home Win","Draw","Away Win"], [home_w, draws, away_w],
                   color=[PALETTE[0],PALETTE[1],PALETTE[2]], edgecolor="white", linewidth=1.5)
for bar, v in zip(bars, [home_w, draws, away_w]):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+5, f"{v}\n({v/n:.0%})",
                 ha="center", va="bottom", fontweight="bold", fontsize=11)
axes[0].set_title("Match outcomes (all WC matches)"); axes[0].set_ylabel("Count")

sns.histplot(wc_df["total_goals"], bins=range(0,14), ax=axes[1], color=PALETTE[3])
axes[1].axvline(wc_df["total_goals"].mean(), color=PALETTE[2], linestyle="--", linewidth=2,
                label=f'Mean = {wc_df["total_goals"].mean():.2f}')
axes[1].set_title("Goals per match distribution"); axes[1].set_xlabel("Goals"); axes[1].legend()

goals_long = pd.melt(wc_df[["home_team_goals","away_team_goals"]].rename(
    columns={"home_team_goals":"Home","away_team_goals":"Away"}))
sns.boxplot(data=goals_long, x="variable", y="value", ax=axes[2],
            hue="variable", palette=[PALETTE[0],PALETTE[2]], legend=False)
axes[2].set_title("Goal distribution: Home vs Away")
axes[2].set_xlabel(""); axes[2].set_ylabel("Goals scored")

plt.suptitle("WC Match Outcomes & Scoring Patterns", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Home win: {home_w/n:.1%}  |  Draw: {draws/n:.1%}  |  Away win: {away_w/n:.1%}")

### 3.5 Top Teams — Goals & Appearances

In [ ]:
home_g = wc_df.groupby("home_team_name")["home_team_goals"].sum()
away_g = wc_df.groupby("away_team_name")["away_team_goals"].sum()
team_g = home_g.add(away_g, fill_value=0).sort_values(ascending=False).head(12)

home_a = wc_df["home_team_name"].value_counts()
away_a = wc_df["away_team_name"].value_counts()
team_a = home_a.add(away_a, fill_value=0).sort_values(ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(x=team_g.values, y=team_g.index, ax=axes[0], color=PALETTE[3])
axes[0].set_title("Top 12 teams by total goals scored"); axes[0].set_xlabel("Goals")
sns.barplot(x=team_a.values, y=team_a.index, ax=axes[1], color=PALETTE[6])
axes[1].set_title("Top 12 teams by WC appearances"); axes[1].set_xlabel("Matches")
plt.suptitle("All-Time WC Team Rankings", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

### 3.6 Scoring Efficiency Scatter (≥10 WC matches)

In [ ]:
team_long = pd.concat([
    wc_df.rename(columns={"home_team_name":"team","home_team_goals":"gf","away_team_goals":"ga"})[["team","gf","ga"]],
    wc_df.rename(columns={"away_team_name":"team","away_team_goals":"gf","home_team_goals":"ga"})[["team","gf","ga"]],
], ignore_index=True)

eff = team_long.groupby("team").agg(matches=("gf","count"),avg_for=("gf","mean"),avg_against=("ga","mean")).reset_index()
eff = eff[eff["matches"] >= 10]

plt.figure(figsize=(13, 8))
sc = plt.scatter(eff["avg_against"], eff["avg_for"], s=eff["matches"]*8, alpha=0.65,
                 c=eff["avg_for"]-eff["avg_against"], cmap="RdYlGn", edgecolors="white")
plt.colorbar(sc, label="Avg goal difference")
for _, row in eff[eff["matches"] >= 25].iterrows():
    plt.annotate(row["team"], (row["avg_against"], row["avg_for"]),
                 fontsize=9, xytext=(5,4), textcoords="offset points")
mgf = eff["avg_for"].mean(); mga = eff["avg_against"].mean()
plt.axhline(mgf, color="gray", linestyle="--", alpha=0.4, label=f"Avg scored={mgf:.2f}")
plt.axvline(mga, color="gray", linestyle=":",  alpha=0.4, label=f"Avg conceded={mga:.2f}")
plt.xlabel("Avg goals conceded"); plt.ylabel("Avg goals scored")
plt.title("Scoring Efficiency — top-left = elite, bottom-right = struggling", fontsize=13)
plt.legend(); plt.tight_layout(); plt.show()

### 3.7 Group Stage vs Knockout

In [ ]:
ko_summary = wc_df.groupby("is_knockout").agg(
    matches=("total_goals","count"), avg_goals=("total_goals","mean"), avg_margin=("margin","mean")
).reset_index()
ko_summary["stage"] = ko_summary["is_knockout"].map({True:"Knockout",False:"Group Stage"})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col, title in zip(axes, ["avg_goals","avg_margin"],
                           ["Avg total goals per match","Avg margin of victory"]):
    bars = ax.bar(ko_summary["stage"], ko_summary[col],
                  color=[PALETTE[0],PALETTE[2]], edgecolor="white", linewidth=1.5)
    for bar, val in zip(bars, ko_summary[col]):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.02, f"{val:.2f}", ha="center", fontweight="bold")
    ax.set_title(title)
plt.suptitle("Group Stage vs Knockout Dynamics", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()
ko_summary[["stage","matches","avg_goals","avg_margin"]]

### 3.8 WC Winners & Tournament Goals

In [ ]:
winners = cups_df["Winner"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
bars = axes[0].barh(winners.index, winners.values, color=PALETTE[6], edgecolor="white")
for bar, v in zip(bars, winners.values):
    axes[0].text(v+0.05, bar.get_y()+bar.get_height()/2, str(int(v)), va="center", fontweight="bold")
axes[0].set_title("World Cup Titles by Country"); axes[0].set_xlabel("Titles"); axes[0].invert_yaxis()
axes[1].plot(cups_df["Year"], cups_df["GoalsScored"], marker="o", color=PALETTE[8], linewidth=2)
axes[1].fill_between(cups_df["Year"], cups_df["GoalsScored"], alpha=0.15, color=PALETTE[8])
axes[1].set_title("Total Goals per World Cup"); axes[1].set_ylabel("Goals scored")
plt.suptitle("WC History at Tournament Level", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

### 3.9 Correlation Heatmap

In [ ]:
num_cols = ["home_team_goals","away_team_goals","total_goals","goal_difference"]
try:
    num_cols += ["half-time_home_goals","half-time_away_goals"]
except: pass
corr = wc_df[[c for c in num_cols if c in wc_df.columns]].corr()
plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, mask=mask, cbar_kws={"shrink":0.8},
            linewidths=0.5, linecolor="white")
plt.title("Feature Correlation Matrix (WC Matches)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

---
## ✅ EDA Complete

**Key takeaways:**
- Home win rate (~47%) dominates — even at a "neutral" tournament, regional crowd support matters
- Average goals/match has declined from ~4.0 (1950s) to ~2.6 (2018+) — football is more tactical now  
- Brazil, Germany, France lead both goals and appearances
- Knockout matches are tighter (lower avg goals, lower margins) than group stage

**→ Next notebook:** `02_feature_engineering.ipynb`
